# chap 1

목차
1. XGBoost란?
2. 데이터 랭글링 (data wrangling)
3. 회귀 모델 제작
4. 분류 모델 제작

## 1. XGBoost란?

앙상블 기법에 대한 이해가 필요하다.
앙상블 기법(Ensemble)은 decision tree같은 방법론의 과대/과소 적합 문제를 해결하기 위해 여러 개의 분류기를 이용해 앙상블을 이루도록 하는 데이터 마이닝 기법이다.
대표적으로 배깅, 부스팅, 랜덤 포레스트 기법이 있다.
- 배깅: 주어진 자료에서 여러 Bootstrap 자료 생성 -> 결합 -> 최종 예측 모형 중 voting을 통해 최종 결과 산출
- 부스팅: 예측력이 약한 모형(Weak Learner)들을 결합 -> 예측 모형 제작
- 랜덤 포레스트: 배깅과 부스팅보다 높은 무작위성 부여해 약한 학습기 생성 -> 선형결합 -> 최종 학습기 제작

XGBoost는 앙상블 기법 중 하나이며, Extreme Gradient Boosting의 약자이다.

## 2. 데이터 랭글링 Data Wrangling

모델링 전의 전처리 단계를 포함하는 용어 -> 로딩, 정제, 분석, 조작

데이터 셋 링크: https://github.com/rickiepark/handson-gb/tree/main/Chapter01

bike_rentals.csv 이거 하나만 다운받으셈

In [ ]:
from pathlib import Path
parent_path = Path.cwd().parents[2] # Path.cwd().parent.parent.parent하고 똑같음. parents는 리스트처럼 접근!
data_dir = parent_path / "04_Resources" / "Dataset" / "XGBoost"

1. 데이터 로딩

데이터셋 설명 (책에 없어서 직접 찾아봄 ㅠㅠ)
- 워싱턴 D.C.의 자전거 공유 시스템(Capital Bikeshare) 데이터셋으로, 머신러닝 회귀(Regression) 모델을 연습할 때 전 세계적으로 가장 많이 쓰이는 표준 데이터 중 하나

1. instant: 데이터 레코드의 고유 인덱스 (1, 2, 3... 순번)
2. dteday: 대여 날짜 (예: 2011-01-01)
3. yr: 연도 (0: 2011년, 1: 2012년)
4. mnth: 월 (1 ~ 12)
5. season: 계절 (1: 봄, 2: 여름, 3: 가을, 4: 겨울)
6. holiday: 공휴일 여부 (0: 휴일 아님, 1: 휴일)
7. weekday: 요일 (0: 일요일, 1: 월요일 ... 6: 토요일)
8. workingday: 평일(근무일) 여부 (주말이나 공휴일이 아니면 1, 쉬는 날이면 0)
9. weathersit: 날씨 상태 (1: 맑음/약간 구름, 2: 안개/흐림, 3: 가벼운 눈/비, 4: 폭우/폭설)
10. temp: 정규화된 실제 섭씨온도
11. atemp: 정규화된 체감 섭씨온도 (바람 등을 고려해 사람이 실제로 느끼는 온도)
12. hum: 정규화된 습도
13. windspeed: 정규화된 풍속
14. casual: 비회원(일회성 이용자)의 자전거 대여 횟수
15. registered: 등록된 회원(정기권 이용자)의 자전거 대여 횟수
16. cnt: 총 자전거 대여 횟수 (Target Variable)

In [110]:
import pandas as pd
df_bikes = pd.read_csv(data_dir / 'bike_rentals.csv')
df_bikes.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1.0,0.0,1.0,0.0,6.0,0.0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1.0,0.0,1.0,0.0,0.0,0.0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1.0,0.0,1.0,0.0,1.0,1.0,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1.0,0.0,1.0,0.0,2.0,1.0,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1.0,0.0,1.0,0.0,3.0,1.0,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


2. 데이터 탐색

In [111]:
df_bikes.describe()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
count,731.000000,731.000000,730.000000,730.000000,731.000000,731.000000,731.000000,731.000000,730.000000,730.000000,728.000000,726.000000,731.000000,731.000000,731.000000
mean,366.000000,2.496580,0.500000,6.512329,0.028728,2.997264,0.682627,1.395349,0.495587,0.474512,0.627987,0.190476,848.176471,3656.172367,4504.348837
std,211.165812,1.110807,0.500343,3.448303,0.167155,2.004787,0.465773,0.544894,0.183094,0.163017,0.142331,0.077725,686.622488,1560.256377,1937.211452
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.059130,0.079070,0.000000,0.022392,2.000000,20.000000,22.000000
25%,183.500000,2.000000,0.000000,4.000000,0.000000,1.000000,0.000000,1.000000,0.336875,0.337794,0.521562,0.134494,315.500000,2497.000000,3152.000000
50%,366.000000,3.000000,0.500000,7.000000,0.000000,3.000000,1.000000,1.000000,0.499166,0.487364,0.627083,0.180971,713.000000,3662.000000,4548.000000
75%,548.500000,3.000000,1.000000,9.750000,0.000000,5.000000,1.000000,2.000000,0.655625,0.608916,0.730104,0.233218,1096.000000,4776.500000,5956.000000
max,731.000000,4.000000,1.000000,12.000000,1.000000,6.000000,1.000000,3.000000,0.861667,0.840896,0.972500,0.507463,3410.000000,6946.000000,8714.000000


In [112]:
df_bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    float64
 3   yr          730 non-null    float64
 4   mnth        730 non-null    float64
 5   holiday     731 non-null    float64
 6   weekday     731 non-null    float64
 7   workingday  731 non-null    float64
 8   weathersit  731 non-null    int64  
 9   temp        730 non-null    float64
 10  atemp       730 non-null    float64
 11  hum         728 non-null    float64
 12  windspeed   726 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(10), int64(5), object(1)
memory usage: 91.5+ KB


3. 결측값 처리

In [113]:
df_bikes.isna().sum().sum() # 결측값 총 개수

np.int64(12)

In [114]:
df_bikes[df_bikes.isna().any(axis=1)] # 결측값이 있는 모든 행
# 행은 0, 열은 1 - 여기서 1이므로 열 방향(세로)를 따라서 누락된 값이 하나 이상인 행을 찾는다.

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
56,57,2011-02-26,1.0,0.0,2.0,0.0,6.0,0.0,1,0.282500,0.282192,0.537917,NaN,424,1545,1969
81,82,2011-03-23,2.0,0.0,3.0,0.0,3.0,1.0,2,0.346957,0.337939,0.839565,NaN,203,1918,2121
128,129,2011-05-09,2.0,0.0,5.0,0.0,1.0,1.0,1,0.532500,0.525246,0.588750,NaN,664,3698,4362
129,130,2011-05-10,2.0,0.0,5.0,0.0,2.0,1.0,1,0.532500,0.522721,NaN,0.115671,694,4109,4803
213,214,2011-08-02,3.0,0.0,8.0,0.0,2.0,1.0,1,0.783333,0.707071,NaN,0.205850,801,4044,4845
298,299,2011-10-26,4.0,0.0,10.0,0.0,3.0,1.0,2,0.484167,0.472846,0.720417,NaN,404,3490,3894
388,389,2012-01-24,1.0,1.0,1.0,0.0,2.0,1.0,1,0.342500,0.349108,NaN,0.123767,439,3900,4339
528,529,2012-06-12,2.0,1.0,6.0,0.0,2.0,1.0,2,0.653333,0.597875,0.833333,NaN,477,4495,4972
701,702,2012-12-02,4.0,1.0,12.0,0.0,0.0,0.0,2,NaN,NaN,0.823333,0.124379,892,3757,4649
730,731,2012-12-31,1.0,NaN,NaN,0.0,1.0,0.0,2,0.215833,0.223487,0.577500,0.154846,439,2290,2729


In [115]:
# 평균값으로 결측치 채우기
df_bikes['hum'] = df_bikes['hum'].fillna(df_bikes['hum'].mean())

In [116]:
# 중간값으로 결측치 채우기
df_bikes['windspeed'] = df_bikes['windspeed'].fillna(df_bikes['windspeed'].median())
df_bikes.loc[[56,81], ['windspeed']] # 56행 81행 windspeed열 출력 loc[[행], [열]] , 모든 열을 보고싶으면 loc[[행]] 이렇게 접근 가능함

,windspeed
56,0.180971
81,0.180971


In [117]:
df_bikes[df_bikes['temp'].isna()] # temp랑 atemp는 결측치가 하나만 있으므로 700행과 702행의 평균으로 채우자

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
701,702,2012-12-02,4.0,1.0,12.0,0.0,0.0,0.0,2,NaN,NaN,0.823333,0.124379,892,3757,4649


In [118]:
fill_list = ['temp', 'atemp']
for i in fill_list:
    tmp = (df_bikes.iloc[700][i] + df_bikes.iloc[702][i])/2  # 평균 계산
    df_bikes[i] = df_bikes[i].fillna(tmp)
df_bikes.iloc[701][['temp', 'atemp']]

temp     0.375417
atemp     0.38635
Name: 701, dtype: object

In [119]:
# 날짜 객체 만들기 (책은 오래된 문법 사용해서 새걸로 바꿈)
df_bikes['dteday'] = pd.to_datetime(df_bikes['dteday'], format='ISO8601') # YYYY-MM-DD hh:mm:ss
df_bikes['dteday']

0     2011-01-01
1     2011-01-02
2     2011-01-03
3     2011-01-04
4     2011-01-05
         ...    
726   2012-12-27
727   2012-12-28
728   2012-12-29
729   2012-12-30
730   2012-12-31
Name: dteday, Length: 731, dtype: datetime64[ns]

In [120]:
import datetime as dt
df_bikes['mnth'] = df_bikes['dteday'].dt.month # 결측치가 하나이기 때문에 그냥 대입해도 되지만 날짜 객체 연습용으로 해봄.
df_bikes.loc[730, 'yr'] = 1.0 # 2011년은 0 2012년은 1로 정규와 되어있기 때문에 그냥 대입해주기 (어차피 1개여서)

In [121]:
# 수치형이 아닌 열 삭제
df_bikes = df_bikes.drop(['dteday'], axis=1) # 날짜 정보는 다른 열에도 있기때문에 삭제해도 됨
df_bikes.isna().sum()

instant       0
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64

In [122]:
df_bikes.info() # 이제 모든 행이 수치형이고 결측값이 없으므로 머신러닝을 시작하자

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   season      731 non-null    float64
 2   yr          731 non-null    float64
 3   mnth        731 non-null    int32  
 4   holiday     731 non-null    float64
 5   weekday     731 non-null    float64
 6   workingday  731 non-null    float64
 7   weathersit  731 non-null    int64  
 8   temp        731 non-null    float64
 9   atemp       731 non-null    float64
 10  hum         731 non-null    float64
 11  windspeed   731 non-null    float64
 12  casual      731 non-null    int64  
 13  registered  731 non-null    int64  
 14  cnt         731 non-null    int64  
dtypes: float64(9), int32(1), int64(5)
memory usage: 82.9 KB


## 3. 회귀 모델 제작

cnt(대여 횟수)를 예측하는 모델을 만들어보자

In [123]:
# casual과 registered는 실전에서 알 수 없는 값이므로 drop
df_bikes = df_bikes.drop(['casual', 'registered'], axis=1)
# 나중을 위해서 전처리 완료한 데이터 저장!
df_bikes.to_csv(data_dir / "bike_rentals_cleaned.csv", index=False) # index=False옵션: 인덱스가 하나의 열로 저장되는것을 방지

머신러닝의 목표는 입력 특성(feature)의 연산을 통해 target feature를 예측하는 것이다.

In [124]:
# 타겟변수 분리
y = df_bikes['cnt'] # 타겟변수는 관습적으로 소문자 y
X = df_bikes.drop('cnt', axis=1) # 입력변수는 관습적으로 대문자 X로 쓴다.

대표적인 회귀 알고리즘은 선형 회귀가 있다.
- **모르면 이거 보셈: https://youtu.be/3dhcmeOTZ_Q?si=SOPUaB1yBU2SjL5J**

모델링을 할때 보통 데이터를 3가지로 나눔
- Train set: 훈련 세트(이걸로 모델 학습함)
- Valid set: train셋으로 만든 모델 테스트, 성능 미세조정(Tuning)
- Test set: 학습과 튜닝이 끝나고 최종적으로 성능 확인

근데 가끔 Train/Test 2단계로만 분리하는데 그 이유는
- 데이터가 너무 적을때
- 교차검증(Cross-Validation)을 사용할 때. (k-fold 등..)
- 단순 선형회귀나 기초 분석처럼 하이퍼파라미터 조정이 따로 필요 없는 모델을 만들때

지금은 단순 선형회귀 모델을 만드므로 2단계로만 분리했음!@

훈련/테스트 세트 나눌땐 사이킷런 라이브러리를 사용함

In [125]:
from sklearn.model_selection import train_test_split # 데이터 분할
from sklearn.linear_model import LinearRegression # 선형 회귀 모델

사이킷런은 다양한 머신러닝 모델을 구현해놓은 라이브러리임.
- **API 레퍼런스: https://scikit-learn.org/stable/api/index.html **

라이브러리들을 사용할 때 AI딸깍도 좋지만 API reference 보면서 어떤 파라미터가 있는지 보는것도 좋음


In [ ]:
# 분할 (이건 그냥 공식임 외우셈)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, # 테스트 데이터를 25%로 설정 -> 자동으로 train_size=0.75
    random_state=2
)

이렇게 분리하는 이유는
- X_train 에 따른 y_train의 변화를 확인하면서 모델을 학습
- 학습한 모델(회귀식)에 X_test 대입 -> y_pred(예측값)
- y_pred(예측)와 y_test(정답)의 오차 정도를 통해 모델 평가

In [131]:
# 모델 학습과 예측은 어떤 모델이든 3단계로 끝난다

# 1. 모델 객체 생성
lin_model = LinearRegression()

# 2. 모델 학습 - X_train에 따른 y_train 관찰
lin_model.fit(X_train, y_train)

# 3. X_test값으로 y_pred 구하기
y_pred_lin = lin_model.predict(X_test)

모델 평가하기

In [132]:
# metrics 클래스에는 다양한 평가 지표가 있다. 그중 mse지표 사용
from sklearn.metrics import mean_squared_error
import numpy as np

mse_lin = mean_squared_error(y_test, y_pred_lin) # !!!!!순서 주의!!!!!, test 먼저 pred 뒤에임!!. test와 pred의 차의 제곱의 평균임

rmse_lin = np.sqrt(mse_lin) # rmse는 mse의 제곱근 씌운 평가지표임
print(f"RMSE: {rmse_lin:.2f}") # 결과 출력: 작을수록 좋은거임!(오차니까)

RMSE: 898.14


In [133]:
df_bikes['cnt'].describe()

count     731.000000
mean     4504.348837
std      1937.211452
min        22.000000
25%      3152.000000
50%      4548.000000
75%      5956.000000
max      8714.000000
Name: cnt, dtype: float64

해석: 표준편차가 1937이다. 

898/1937 = 20% 정도의 표준 오차를 가지네!

데이터의 아무런 정보 없이 데이터의 변동성을 1900 -> 900 정도로 줄였다!!

이제 **XGBRegressor**을 사용해보자

아나콘다 패키지에 안깔려있음(아마도) 

pip install xgboost ㄱㄱ

In [135]:
# sklearn이 머신러닝 생태계를 독식해서 xgboost도 sklearn과 모델링 방식을 비슷하게 통일하는 방향으로 업데이트함!
from xgboost import XGBRegressor
xgbr_model = XGBRegressor() # 1. 모델 객체 생성
xgbr_model.fit(X_train, y_train) # 2. 학습
y_pred_xgbr = xgbr_model.predict(X_test) # 3. 예측

mse_xgbr = mean_squared_error(y_test, y_pred_xgbr)
rmse_xgbr = np.sqrt(mse_xgbr)
print(f"RMSE: {rmse_xgbr:.2f}")

RMSE: 695.54


898 -> 695로 모델만 바꿨을 뿐인데 오차가 엄청나가 줄음 ㄷㄷ ^오^

그 이유는 나아아중에 정리할 예정임

그런데 train/test 세트 분할 방법이 달라지면 결과도 불안정하게 바뀔 수 있음

실제로 train_test_split 함수에서 random_state 값(난수)을 바꾸면 결과도 달라짐!

그래서 교차검증(cross-validation)을 하는데 대표적으로 k-fold 교차 검증 방법이 있음


모르면 보고오셈: **https://youtu.be/k4mZS-YWpdk?si=kF8NdIW2Zzo1qN1y**

일반적으로 k는 3,4,5,10 사용함. 반복마다 테스트 세트는 중복되지 않음

In [150]:
from sklearn.model_selection import cross_val_score # 사이킷런에 이미 구현되어있음 캬

# 과정
# 1. 모델 생성
model = LinearRegression()
scores = cross_val_score(model, X, y,
                        scoring='neg_root_mean_squared_error', # neg(음수): sklearn은 기본적으로 높은 점수를 좋은거라고 생각하므로 오차는 그와 반대인 음수 사용
                        cv=10)
rmse_list = -scores # - 붙여서 원래 값으로 (ndarray 자료형임)
print(f"회귀 rmse: {np.round(rmse_list, 2)}")
print(f"평균 rmse: {rmse_list.mean():.2f}")

회귀 rmse: [ 504.29  840.24 1140.99  728.24  640.14  969.33 1133.16 1252.96 1084.67
 1425.49]
평균 rmse: 971.95


In [151]:
model_2 = XGBRegressor()
scores = cross_val_score(model_2, X, y,
                        scoring='neg_root_mean_squared_error', # neg(음수): sklearn은 기본적으로 높은 점수를 좋은거라고 생각하므로 오차는 그와 반대인 음수 사용
                        cv=10)
rmse_list_2 = -scores # - 붙여서 원래 값으로 (ndarray 자료형임)
print(f"회귀 rmse: {np.round(rmse_list_2, 2)}")
print(f"평균 rmse: {rmse_list_2.mean():.2f}")

회귀 rmse: [ 825.06  890.79  575.63  679.82  894.34  880.11  904.39  817.98  772.35
 1961.02]
평균 rmse: 920.15


XGBRegressor가 좀 더 뛰어나다고 볼 수 있다

## 4. 분류 모델 제작

분류는 범주형 출력을 예측하는 머신러닝 알고리즘이다.

- 0 1
- 예 아니오
- 빨강 주황 노랑 초록

이와 같은 출력을 예측한다.

대표적으로 로지스틱 회귀 모델(logistic regression)이 있다 (이름만 회귀고 분류모델임 ㅇㅇ)

딥러닝에서도 자주 쓰이는 시그모이드 함수를 이용함

과정
1. w(가중치) b(편향)의 초깃값 설정
2. x를 흘려 보내 z값을 구하고 시그모이드 함수에 통과시켜 확률 벡터 도출
3. 정답값과 비교 ex) 정답: [0, 0, 0, 1] vs 예측 [0.2, 0.3, 0,1, 0.4]
4. 경사하강법 등 최적화 방법을 통해 적절하게 w와 b 수정
5. 반복

$$z = wx + b$$

$$Output = Sigmoid(z) = \frac{1}{1 + e^{-z}}$$

데이터셋은 UCI 머신러닝 저장소 웹사이트에 있는 인구 조사 소득 데이터셋을 사용한다

In [153]:
df_census = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data') # 이거 복붙하셈
df_census.head()

,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
0,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
1,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
2,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
3,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
4,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


첫 번째 행이 열 제목으로 들어가버림 샤갈;

이때는 아래와 같이 로드하면 해결됨

In [154]:
df_census = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data', header=None)
df_census.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [156]:
# 열 이름은 다음과 같다
df_census.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation',
                  'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
                   'income']
df_census.head() # 깔끔하제>

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [161]:
df_census.info() # 결측치가 없음, object 타입이 몇 개 있음

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [162]:
df_census = df_census.drop(['education'], axis=1) # education_num과 중복되므로 제거

범주형 데이터를 연산할 수 없으므로 원-핫 인코딩을 해주자

원-핫 인코딩 예시
- 과일 컬럼에 사과, 포도, 바나나 범주가 있다
- 원-핫 인코딩을 하면 사과, 포도, 바나나 컬럼을 추가한다
- (사과 1 포도 0 바나나 0) 또는 (사과 0 포도 1 바나나 0) 또는 (사과 0 포도 0 바나나 1)

In [163]:
# pd.get_dummies함수 사용
df_census = pd.get_dummies(df_census)
df_census.head()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_ ?,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,...,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,income_ <=50K,income_ >50K
0,39,77516,13,2174,0,40,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
1,50,83311,13,0,0,13,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,38,215646,9,0,0,40,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
3,53,234721,7,0,0,40,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
4,28,338409,13,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


타겟 데이터 분리를 해보자. 소득이 50k를 넘는지 아닌지를 **분류**할거임

In [165]:
df_census = df_census.drop('income_ <=50K', axis=1) # 원핫 인코딩때문에 같은 내용을 담은 열이 2개이므로 하나 제거

In [166]:
y = df_census['income_ >50K']
X = df_census.drop('income_ >50K', axis=1)

In [ ]:
from sklearn.linear_model import LogisticRegression

# 교차 검증 함수 생성
def cross_val(classifier, num_splits=10):
    model = classifier
    scores = cross_val_score(model, X, y, cv=num_splits) # 사이킷런에서 회귀 모델의 기본 평가 지표는 결정계수(R2)이므로 따로 scoring 매개변수 필요엄슴
    print(f'정확도: {np.round(scores,2)}')               # R^2 = 1 - (SSE/SST) 이다. SST는 실제값-평균값, SSE는 실제값-모델예측값
    print(f'평균 정확도{scores.mean():.2f}')             # 즉, 내가 줄이지 못한 오차의 비율을 뺀 값이다.
    
cross_val(LogisticRegression(max_iter=1000)) # 막 뭐가 길게 나오면 아마도 경사하강법 바닥을 못찾아서 그런걸꺼임. 기본 100회 반복인데 1000회로 설정
                                             # 오류는 아니긴 함. max_iter를 늘리거나 데이터 스케일링을 해주면 어느정도 해결 가능함. 자세한건 gpt 제미나이 grok let's go

c:\Users\devsa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\devsa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

정확도: [0.84 0.84 0.84 0.83 0.84 0.83 0.84 0.84 0.84 0.84]
평균 정확도0.84


c:\Users\devsa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


이제 XGBClassifier를 사용해보자

In [170]:
from xgboost import XGBClassifier

cross_val(XGBClassifier(n_estimators=5)) # 오차를 줄이는 나무를 순차적으로 5개 만들겠다는 뜻. 나중에 자세하게 다룰 예정

정확도: [0.85 0.86 0.86 0.85 0.86 0.86 0.86 0.87 0.87 0.86]
평균 정확도0.86


XGBClassifier가 연산량이 압도적으로 적은데 더 나은 결과를 보여줌 ㄷㄷㄷㄷㄷㄷ;;;

Chapter 2에서는 결정 트리를 만들고 성능을 높이는 하이퍼파라미터 튜닝을 할거임